# BIU DS23 · Module 5 · The Grounded Assistant Assignment · Starter Notebook
**Due September 17. Read the assignment document first, then work here.**

This notebook is scaffolding, not a solution. Your job is to build one grounded
research assistant over arXiv abstracts in YOUR capstone domain, and to JUSTIFY
every decision. Remember the rubric: **70% of the grade is on your reasoning, 20%
on technical correctness, 10% on the quality of your evaluation set.** A modest or
even negative improvement with excellent justification beats a high number you
cannot explain.

**One track, personalized.** Everyone builds the same kind of system, but on their
own corpus: at least 300 arXiv abstracts in the domain of your capstone project.
You write your own evaluation set, lock it, and measure groundedness before and
after one improvement of your choice.

**The one rule that never bends:** every answer the assistant gives must be
GROUNDED in the retrieved abstracts and must CITE their arXiv IDs. When the answer
is not in the corpus, the assistant must say "I don't know" rather than invent one.
An ungrounded, confident answer is exactly the failure this whole module warns
against, and it caps your grade.

**Submission is via Git.** You push this notebook, your saved corpus, and your
decisions report to your course Git repository. Two consequences follow from that,
and both are graded:
1. The notebook must run top to bottom from a clean checkout.
2. **Never paste an API key into a cell.** A key committed to Git is a leaked key.
   Read keys only through Colab Secrets, exactly as shown in section 2. A visible
   key in your submission is a security defect, and it is the same lesson as
   prompt injection: what happens when your system meets the outside world.

## 0 · Setup

In [41]:
!pip install -q feedparser sentence-transformers chromadb openai

import os, re, json, time, random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Optional: mount Drive to save your corpus so runs are reproducible.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/DS23/module5/"
    os.makedirs(SAVE_DIR, exist_ok=True)
except Exception:
    SAVE_DIR = "./"
print("save dir:", SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
save dir: /content/drive/MyDrive/DS23/module5/


## 1 · Build your corpus (given scaffolding)
Fetch at least 300 arXiv abstracts in your capstone domain. This fetch code is
given; the only thing you change is the query, so the corpus is yours. Abstracts
and metadata only, never full PDFs: the abstracts and metadata are freely
reusable, PDF licenses vary paper by paper.

In [42]:
import urllib.parse
import feedparser

def fetch_arxiv(query, total=300, page=100):
    """Fetch at least `total` arXiv abstracts for a search query."""
    base = "http://export.arxiv.org/api/query"
    records, start = [], 0
    while len(records) < total:
        params = urllib.parse.urlencode({
            "search_query": query,
            "start": start,
            "max_results": page,
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        })
        feed = feedparser.parse(f"{base}?{params}")
        if not feed.entries:
            break
        for e in feed.entries:
            records.append({
                "id": e.id.split("/abs/")[-1],
                "title": " ".join(e.title.split()),
                "abstract": " ".join(e.summary.split()),
                "primary_category": e.arxiv_primary_category["term"],
                "categories": [t["term"] for t in e.tags],
                "year": int(e.published[:4]),
            })
        start += page
        time.sleep(3)   # arXiv asks for 3 seconds between calls
    return records[:total]

QUERY = '(abs:"skin lesion" OR abs:melanoma OR abs:mole) AND (abs:classification OR abs:mapping OR abs:segmentation) AND (cat:cs.CV OR cat:eess.IV)'


papers = fetch_arxiv(QUERY, total=300)
print(f"fetched {len(papers)} abstracts")
assert len(papers) >= 300, "widen your query (add a category with OR, or relax the term)"

# Save the corpus so every later run is on the SAME data.
with open(SAVE_DIR + "corpus.json", "w") as f:
    json.dump(papers, f)
print("saved corpus.json")

fetched 300 abstracts
saved corpus.json


## 2 · The LLM backend (given)
One backend switch and one `chat()` wrapper, with a saved-response fallback so a
dropped API call never stalls your run. Keys are read ONLY through Colab Secrets.
Do not edit this cell except to switch backend or model.

In [43]:
def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)          # keys live in Colab Secrets, never in a cell
    except Exception:
        return os.environ.get(name)

BACKEND    = "groq"
GROQ_MODEL = "openai/gpt-oss-120b"          # current free-tier production model
SAVED = {}
USE_SAVED = False

def chat(messages, temperature=0.0, max_tokens=512, seed=SEED):
    from openai import OpenAI
    client = OpenAI(api_key=get_secret("GROQ_API_KEY"),
                    base_url="https://api.groq.com/openai/v1")
    extra_body = {}
    # gpt-oss is a reasoning model on Groq: hide the trace and leave room for the
    # answer, since reasoning tokens are charged against the output budget.
    if "gpt-oss" in GROQ_MODEL:
        extra_body["reasoning_format"] = "hidden"
        max_tokens = max(max_tokens, 1024)
    elif "qwen3" in GROQ_MODEL:
        extra_body["reasoning_effort"] = "none"
    resp = client.chat.completions.create(
        model=GROQ_MODEL, messages=messages, temperature=temperature,
        max_tokens=max_tokens, seed=seed, extra_body=extra_body)
    return (resp.choices[0].message.content or "").strip()

def ask(messages, label, **kw):
    if USE_SAVED and label in SAVED:
        return SAVED[label]
    try:
        return chat(messages, **kw)
    except Exception as e:
        if label in SAVED:
            print(f"[fallback] {type(e).__name__}, using saved response")
            return SAVED[label]
        raise

# quick warm-up so you find out in the first minute that the key is wired.
print("warm-up:", ask([{"role": "user", "content": "Reply with one word: READY"}],
                      label="warmup", max_tokens=50))

warm-up: READY


In [44]:
# קוד קטן שיציג לך את 5 המאמרים הראשונים ששמרת בקורפוס
import json

# טעינת הקובץ ששמרת בשלב הראשון (אם השתמשת בנתיב אחר, שנו את ה-SAVE_DIR)
with open(SAVE_DIR + "corpus.json", "r") as f:
    loaded_papers = json.load(f)

# הדפסת המאמרים הראשונים כדי שתוכלו לקרוא ולחבר שאלות
for i, paper in enumerate(loaded_papers[:5]):
    print(f"--- מאמר מספר {i+1} ---")
    print(f"כותרת (Title): {paper['title']}")
    print(f"תקציר (Abstract): {paper['abstract']}\n")

--- מאמר מספר 1 ---
כותרת (Title): A Comparative Evaluation of Pre-trained Convolutional Neural Networks for Melanoma Detection
תקציר (Abstract): Early diagnosis of melanoma is critical for improving patient survival rates. However, accurately distinguishing melanoma from other skin lesions remains a significant clinical challenge due to the high visual similarity among lesion types and variability in image acquisition conditions. Artificial intelligence, particularly machine learning, has emerged as a promising tool to support dermatological diagnosis by automating feature extraction from medical images. Among the available approaches, convolutional neural networks (CNNs) have demonstrated strong performance in image classification tasks, making them well-suited for analyzing both dermatoscopic and histopathological images, given their ability to capture hierarchical visual patterns relevant to lesion characterization. Nevertheless, despite numerous pre-trained CNN architectures havin

## 3 · The frozen evaluation set (YOUR work, then locked)
Write 20 questions about your corpus. **15 must be answerable from the abstracts,
and 5 must NOT be answerable from your corpus** (out-of-scope questions that test
whether the assistant refuses instead of inventing). Once written, this set is
FROZEN: you measure before and after your improvement on the exact same 20
questions, or the comparison is not fair.

Quality of this set is 10% of your grade. Good in-scope questions have a clear
answer in a specific abstract; good out-of-scope questions are plausible but
genuinely absent from your corpus.

In [45]:
# TODO 3: write your evaluation set. Each item: the question, and whether the
# answer exists in your corpus (True) or not (False). Aim for 15 True, 5 False.
EVAL = [
    # --- שאלות מתוך מאמר 1 (True) ---
    {"q": "Which datasets were evaluated in the comparative study of pre-trained CNNs for melanoma detection?", "in_corpus": True},
    {"q": "What was the highest accuracy achieved by ResNet50 on the HAM10000 dataset in the pre-trained CNN evaluation study?", "in_corpus": True},
    {"q": "What accuracy did InceptionV3 achieve on the ISIC 2018 dataset according to the comparative evaluation?", "in_corpus": True},
    {"q": "Which specific CNN architectures were tested under the same training protocol for skin lesion classification?", "in_corpus": True},
    {"q": "Does model performance differ between dermatoscopic and histopathological image modalities according to the comparative evaluation?", "in_corpus": True},

    # --- שאלות מתוך מאמר 2 (True) ---
    {"q": "What is the name of the cross-domain transfer framework proposed for bruise segmentation?", "in_corpus": True},
    {"q": "On which skin lesion dataset was the BruNet framework trained?", "in_corpus": True},
    {"q": "Which baseline models did BruNet outperform, including ChatGPT-assisted versions and medical models?", "in_corpus": True},

    # --- שאלות מתוך מאמר 3 (True) ---
    {"q": "What is the name of the label-free post-conformal decision rule proposed to handle transitional uncertainty?", "in_corpus": True},
    {"q": "By how much did the overall accuracy improve on the ISIC skin lesion benchmark when using AdaConRed?", "in_corpus": True},
    {"q": "What specific encoder was used to provide embeddings in the AdaConRed five-stage pipeline?", "in_corpus": True},
    {"q": "What is the URL of the code repository provided for AdaConRed?", "in_corpus": True},

    # --- שאלות מתוך מאמר 4 (True) ---
    {"q": "What is the name of the Compositional Reward Model framework proposed for conditional medical image generation?", "in_corpus": True},
    {"q": "By what percentage did PRISM increase the ISIC F1 score for downstream models?", "in_corpus": True},

    # --- שאלות מתוך מאמר 5 (True) ---
    {"q": "Which specific deep learning backbone architecture was used to build the attention-guided global and local fusion framework?", "in_corpus": True},

    # --- שאלות מחוץ לקורפוס - הגיוניות לתחום אבל לא קיימות במאמרים האלו (False) ---
    {"q": "What is the recommended legal guidelines for deploying skin cancer detection models in German clinics?", "in_corpus": False},
    {"q": "How much Nvidia V100 GPU memory is required to run the BruNet inference pipeline on a smartphone?", "in_corpus": False},
    {"q": "What is the specific patient survival rate after 5 years when using the AdaConRed framework in hospitals?", "in_corpus": False},
    {"q": "Does the PRISM framework support the generation of 3D MRI scans of brain tumors?", "in_corpus": False},
    {"q": "What is the cost in USD for annotating one single image in the CR-AI4SkIN dataset?", "in_corpus": False}
]

# בדיקות התקינות של המרצה - יוודאו שהכל בול במידות הנכונות
assert len(EVAL) == 20, "write exactly 20 questions"
assert sum(1 for e in EVAL if not e["in_corpus"]) == 5, "exactly 5 must be out-of-scope"
print("eval set locked:", len(EVAL), "questions,",
      sum(e["in_corpus"] for e in EVAL), "in-corpus,",
      sum(not e["in_corpus"] for e in EVAL), "out-of-scope")


eval set locked: 20 questions, 15 in-corpus, 5 out-of-scope


## 4 · The groundedness metric (given)
Groundedness is the fraction of an answer's cited arXiv IDs that actually appear
in the retrieved context. It is the number you will improve. This deterministic
proxy is given so everyone measures the same way; you MAY add a stricter
LLM-as-judge version in your improvement, and justify it.

In [51]:
def groundedness(answer, retrieved_ids):
    """Fraction of arXiv IDs cited in the answer that were actually retrieved.
    Returns a value in [0, 1]. An answer that cites nothing scores 0."""
    # Updated regex to include optional 'v' and version number (e.g., 2401.01234v1)
    cited = set(re.findall(r"\d{4}\.\d{4,5}(?:v\d+)?", answer))
    if not cited:
        return 0.0
    supported = cited & set(retrieved_ids)
    return len(supported) / len(cited)

# Deterministic sanity checks on the metric itself (do not change these).
assert groundedness("See 2401.01234 and 2312.09999.", ["2401.01234", "2312.09999"]) == 1.0
assert groundedness("See 2401.01234v1 and 2312.09999v2.", ["2401.01234v1", "2312.09999v2"]) == 1.0 # Added test case for versioned IDs
assert groundedness("See 2401.01234 and 9999.99999.", ["2401.01234"]) == 0.5
assert groundedness("No citation here.", ["2401.01234"]) == 0.0
print("groundedness metric verified")

groundedness metric verified


---
# Your work starts here
Each section has a TODO. Fill it, and record your decision and reasoning in the
justification template (assignment document). One line of code is not enough; the
WHY is what is graded.

## 5 · Chunk, embed, and index
Long abstracts may need chunking; short ones do not. Choose a chunking strategy
and justify it, then embed and index in Chroma with metadata (category, year, id)
so you can filter later.

In [48]:
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# TODO 5: decide your chunking strategy and justify it. For abstracts, whole-text
# is often fine; justify whether you chunk at all, and if so, size and overlap.
def to_chunks(text):
    return [text]   # <- replace with your justified strategy if you chunk

records = []
for p in papers:
    for j, ch in enumerate(to_chunks(p["abstract"])):
        records.append({"id": f"{p['id']}::c{j}", "paper_id": p["id"], "text": ch,
                        "category": p["primary_category"], "year": p["year"]})

emb = embedder.encode([r["text"] for r in records],
                      normalize_embeddings=True, show_progress_bar=True)
client = chromadb.Client()
col = client.get_or_create_collection("arxiv") # Changed to get_or_create_collection
col.add(ids=[r["id"] for r in records],
        documents=[r["text"] for r in records],
        metadatas=[{"paper_id": r["paper_id"], "category": r["category"],
                    "year": r["year"]} for r in records],
        embeddings=[e.tolist() for e in emb])
print("indexed chunks:", col.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

indexed chunks: 300


## 6 · Retrieve and answer, with citations
Retrieve the top passages for a question, then have the model answer USING ONLY
those passages, and cite the arXiv IDs it used. This grounded-answer prompt is the
heart of the assignment: write it so the model refuses when the context does not
contain the answer.

In [50]:
def retrieve(question, k=5, where=None):
    q = embedder.encode([question], normalize_embeddings=True)[0].tolist()
    res = col.query(query_embeddings=[q], n_results=k, where=where)
    return res["documents"][0], res["metadatas"][0]

# TODO 6: write the grounded-answer system prompt. It MUST instruct the model to
# use only the provided passages, cite the arXiv IDs it relied on, and reply
# exactly "I don't know" when the passages do not contain the answer.
GROUNDED_SYSTEM = (
    "You are an AI research assistant evaluating scientific papers on skin cancer. "
    "Your objective is to answer the query using EXCLUSIVELY the provided CONTEXT text.\n"
    "Strict Guidelines:\n"
    "1. Never use outside information or make assumptions. Stick only to what is written.\n"
    "2. Every time you state a fact from a paper, include its arXiv ID at the end of the sentence (like [2401.01234]).\n"
    "3. If the answer cannot be found within the provided context, or if the data is insufficient, your entire response must be exactly: I don't know. Do not add any introduction, explanation, or punctuation."
)


def answer(question, k=5):
    docs, metas = retrieve(question, k=k)
    context = "\n\n".join(f"[{m['paper_id']}] {d}" for d, m in zip(docs, metas))
    out = ask([{"role": "system", "content": GROUNDED_SYSTEM},
               {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}],
              label=f"ans_{abs(hash(question))%10000}")
    return out, [m["paper_id"] for m in metas]

# Try one in-corpus question to see grounding + citation in action.
a, ids = answer(EVAL[0]["q"])
print(a)

The study evaluated the HAM10000, ISIC 2018, and CR‑AI4SkIN datasets [2609.11550v1].


## 7 · The "I don't know" test
Run your 5 out-of-scope questions. A correct system refuses all 5. Report how many
it refused, and inspect any it answered anyway, that is a grounding failure worth
analyzing in your report.

In [52]:
# TODO 7: run the out-of-scope questions and count refusals.
def is_refusal(text):
    return "i don't know" in text.lower() or "i do not know" in text.lower()
out_of_scope = [e for e in EVAL if not e["in_corpus"]]
refused = sum(is_refusal(answer(e["q"])[0]) for e in out_of_scope)
print(f"refused {refused} of {len(out_of_scope)} out-of-scope questions")



refused 5 of 5 out-of-scope questions


## 8 · Baseline groundedness (before)
Measure groundedness on the 15 in-corpus questions with your current system, and
LOCK this number. This is your "before".

In [53]:
# TODO 8: compute baseline groundedness over the in-corpus questions.
def mean_groundedness():
    scores = []
    for e in EVAL:
        if not e["in_corpus"]:
            continue
        ans, ids = answer(e["q"])
        scores.append(groundedness(ans, ids))
    return float(np.mean(scores)) if scores else 0.0
baseline = mean_groundedness()
print("baseline groundedness:", round(baseline, 3))

baseline groundedness: 0.533


In [54]:
# נריץ בדיקה על 3 השאלות הראשונות ונראה בדיוק מה המודל עונה
in_corpus_questions = [e for e in EVAL if e["in_corpus"]][:3]

for i, e in enumerate(in_corpus_questions):
    print(f"=== שאלה מספר {i+1} ===")
    print(f"השאלה: {e['q']}")

    # הפעלת המערכת לקבלת תשובה
    ans, retrieved_ids = answer(e["q"])

    print(f"תשובת המודל והציטוטים:\n{ans}")
    print(f"ציון המבוססות לשאלה זו: {groundedness(ans, retrieved_ids)}")
    print("-" * 50)


=== שאלה מספר 1 ===
השאלה: Which datasets were evaluated in the comparative study of pre-trained CNNs for melanoma detection?
תשובת המודל והציטוטים:
The study evaluated the HAM10000, ISIC 2018, and CR‑AI4SkIN datasets [2609.11550v1].
ציון המבוססות לשאלה זו: 1.0
--------------------------------------------------
=== שאלה מספר 2 ===
השאלה: What was the highest accuracy achieved by ResNet50 on the HAM10000 dataset in the pre-trained CNN evaluation study?
תשובת המודל והציטוטים:
84% [2609.11550v1]
ציון המבוססות לשאלה זו: 1.0
--------------------------------------------------
=== שאלה מספר 3 ===
השאלה: What accuracy did InceptionV3 achieve on the ISIC 2018 dataset according to the comparative evaluation?
תשובת המודל והציטוטים:
I don't know
ציון המבוססות לשאלה זו: 0.0
--------------------------------------------------


## 9 · One improvement of your choice
Pick ONE improvement and justify it: smarter chunking, reranking (retrieve wide
then rank narrow), a metadata filter (by category or year), or a stricter grounded
prompt. Change only that one thing, so the before/after comparison isolates its
effect.

In [55]:
# TODO 9: implement your single improvement here, then rebuild `answer`/`retrieve`

# 1. פונקציית שליפה משופרת - מעלה את k ל-15 כדי לא לפספס מאמרים
def retrieve(question, k=15, where=None):
    q = embedder.encode([question], normalize_embeddings=True)[0].tolist()
    res = col.query(query_embeddings=[q], n_results=k, where=where)
    # תיקון קטן בשליפת הרשימות כדי למנוע באגים בקולאב
    return res["documents"][0], res["metadatas"][0]

# 2. בנייה מחדש של פונקציית התשובות עם ה-k המורחב
def answer(question, k=15):
    docs, metas = retrieve(question, k=k)
    context = "\n\n".join(f"[{m['paper_id']}] {d}" for d, m in zip(docs, metas))
    out = ask([{"role": "system", "content": GROUNDED_SYSTEM},
               {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}],
              label=f"ans_{abs(hash(question))%10000}")
    return out, [m["paper_id"] for m in metas]

print("Improvement implemented: Expanded retrieval context length (k=15)")


Improvement implemented: Expanded retrieval context length (k=15)


## 10 · Groundedness after, and the lift
Measure again on the SAME 15 in-corpus questions, and report before, after, and the
lift, honestly. A small or negative lift is a valid result: explain what it tells
you about your corpus, your retrieval, or your prompt.

In [56]:
# TODO 10: measure after your improvement and report the lift.
after = mean_groundedness()
print(f"before : {baseline:.3f}")
print(f"after  : {after:.3f}")
print(f"lift   : {after - baseline:+.3f}")

before : 0.533
after  : 0.733
lift   : +0.200


---
## Before you submit
- Every answer is grounded and cites its arXiv IDs; out-of-scope questions get
  "I don't know". Check twice.
- Your evaluation set has exactly 20 questions, 5 of them out-of-scope, and it was
  frozen before you measured.
- The justification template is filled for every decision: what, why, alternative
  rejected, evidence.
- Before and after groundedness are reported honestly, whatever the lift.
- **No API key appears anywhere in the notebook.** Keys come only from Colab
  Secrets. A key committed to Git is a leaked key.
- The notebook runs top to bottom from a clean checkout, then you **push to your
  course Git repository**: this notebook, `corpus.json`, and your decisions report.

Good luck. The reasoning is the point.